# Vazamento de rótulo e atalho pela `Flgs` no WUSTL-EHMS-2020

Este notebook analisa só duas questões do dataset:

1. **Vazamento de rótulo por colunas.** Alguma coluna do CSV entrega o rótulo sozinha?
2. **Atalho (shortcut learning) pela coluna `Flgs`.** Quanto do desempenho no Spoofing vem de uma marca do Argus que aparece no começo de cada ataque?

O modelo é sempre um `HistGradientBoostingClassifier(random_state=0)`. As métricas são AUROC e AUPRC.

## 1. Carregar os dados

In [1]:
import numpy as np
import polars as pl
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(40)

df = pl.read_csv("_local/data/wustl-ehms-2020_with_attacks_categories.csv", infer_schema_length=None)
print("shape:", df.shape)
print(df["Label"].value_counts().sort("Label"))
print(df["Attack Category"].value_counts().sort("count", descending=True))

shape: (16318, 45)
shape: (2, 2)
┌───────┬───────┐
│ Label ┆ count │
│ ---   ┆ ---   │
│ i64   ┆ u32   │
╞═══════╪═══════╡
│ 0     ┆ 14272 │
│ 1     ┆ 2046  │
└───────┴───────┘
shape: (3, 2)
┌─────────────────┬───────┐
│ Attack Category ┆ count │
│ ---             ┆ ---   │
│ str             ┆ u32   │
╞═════════════════╪═══════╡
│ normal          ┆ 14272 │
│ Spoofing        ┆ 1124  │
│ Data Alteration ┆ 922   │
└─────────────────┴───────┘


## 2. Features oficiais e colunas extras

O artigo do dataset (Hady et al., "Intrusion Detection System for Healthcare Systems Using Medical and Network Data: A Comparison Study", IEEE Access 8, 2020, DOI 10.1109/ACCESS.2020.3000421) lista 34 features na Tabela 1. No CSV, `Heart_rate` vem com r minúsculo.

Conferimos que as 34 existem, quais são constantes e quais colunas do CSV ficam fora da Tabela 1.

In [2]:
OFICIAIS_34 = (
    "SrcBytes DstBytes SrcLoad DstLoad SrcGap DstGap SIntPkt DIntPkt SIntPktAct DIntPktAct "
    "SrcJitter DstJitter sMaxPktSz dMaxPktSz sMinPktSz dMinPktSz Dur Trans TotPkts TotBytes "
    "Loss pLoss pSrcLoss pDstLoss Rate Load Temp SpO2 Pulse_Rate SYS DIA Heart_rate Resp_Rate ST"
).split()
assert len(OFICIAIS_34) == 34
assert all(c in df.columns for c in OFICIAIS_34), "falta feature oficial no CSV"

constantes = [c for c in OFICIAIS_34 if df[c].n_unique() == 1]
FEATURES = [c for c in OFICIAIS_34 if c not in constantes]
extras = [c for c in df.columns if c not in OFICIAIS_34 + ["Attack Category", "Label"]]

print("oficiais constantes:", constantes)
print("oficiais úteis:", len(FEATURES))
print("colunas fora da Tabela 1:", extras)

oficiais constantes: ['SrcGap', 'DstGap', 'DIntPktAct', 'dMinPktSz', 'Trans']
oficiais úteis: 29
colunas fora da Tabela 1: ['Dir', 'Flgs', 'SrcAddr', 'DstAddr', 'Sport', 'Dport', 'SrcMac', 'DstMac', 'Packet_num']


## 3. Teste de vazamento coluna por coluna

Para cada coluna medimos duas coisas, com split aleatório estratificado 70/30:

- `linhas_puras`: fração das linhas cujo valor naquela coluna só aparece com um rótulo. Perto de 1 quer dizer que a coluna funciona como chave do rótulo (ou como ID).
- `auroc_sozinha`: AUROC no teste de um gradient boosting treinado só com aquela coluna. Strings viram códigos categóricos.

Uma feature honesta pode ter AUROC alta. O sinal de vazamento é a coluna ser um identificador ou um artefato da coleta, e não uma medida do tráfego ou do paciente.

In [3]:
y = df["Label"].to_numpy()
idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=0.3, stratify=y, random_state=0)


def como_numero(col):
    s = df[col]
    if s.dtype == pl.String:
        return s.cast(pl.Categorical).to_physical().to_numpy().astype(float)
    return s.to_numpy().astype(float)


def auroc_aleatorio(X):
    modelo = HistGradientBoostingClassifier(random_state=0).fit(X[idx_tr], y[idx_tr])
    return roc_auc_score(y[idx_te], modelo.predict_proba(X[idx_te])[:, 1])


resultados = []
for col in extras + FEATURES:
    rotulos_por_valor = df.group_by(col).agg(pl.col("Label").n_unique().alias("n_rotulos"))
    puras = df.join(rotulos_por_valor, on=col, how="left")["n_rotulos"].eq(1).mean()
    resultados.append({
        "coluna": col,
        "oficial": col in FEATURES,
        "valores_distintos": df[col].n_unique(),
        "linhas_puras": round(puras, 3),
        "auroc_sozinha": round(auroc_aleatorio(como_numero(col)[:, None]), 3),
    })

vazamento = pl.DataFrame(resultados).sort("auroc_sozinha", descending=True)
print(vazamento)

shape: (38, 5)
┌────────────┬─────────┬───────────────────┬──────────────┬───────────────┐
│ coluna     ┆ oficial ┆ valores_distintos ┆ linhas_puras ┆ auroc_sozinha │
│ ---        ┆ ---     ┆ ---               ┆ ---          ┆ ---           │
│ str        ┆ bool    ┆ i64               ┆ f64          ┆ f64           │
╞════════════╪═════════╪═══════════════════╪══════════════╪═══════════════╡
│ SrcMac     ┆ false   ┆ 2                 ┆ 1.0          ┆ 1.0           │
│ Packet_num ┆ false   ┆ 16306             ┆ 1.0          ┆ 0.853         │
│ Sport      ┆ false   ┆ 16314             ┆ 1.0          ┆ 0.852         │
│ DstJitter  ┆ true    ┆ 6046              ┆ 0.779        ┆ 0.849         │
│ DIntPkt    ┆ true    ┆ 6094              ┆ 0.774        ┆ 0.839         │
│ SrcLoad    ┆ true    ┆ 8353              ┆ 0.835        ┆ 0.811         │
│ SIntPkt    ┆ true    ┆ 8505              ┆ 0.835        ┆ 0.811         │
│ Dur        ┆ true    ┆ 8504              ┆ 0.835        ┆ 0.811        

Três colunas chamam atenção.

- **`SrcMac`** separa as classes sozinha. O próprio artigo do dataset diz como o rótulo foi feito: "Source MAC address is used to label the data". É o rótulo com outro nome.
- **`Packet_num`** parece um contador de linha. Abaixo medimos a correlação com a posição no arquivo. Ela é alta sozinha porque os ataques acontecem em trechos contíguos do arquivo.
- **`Sport`** tem quase um valor por linha. Funciona como ID, então não generaliza.

`Dir`, `SrcAddr`, `DstAddr`, `DstMac` e `Dport` são constantes. Entre as 29 oficiais, a melhor sozinha é `DstJitter` (0,849). `Flgs` dá 0,75 e tem só 0,9% de linhas puras, então parece inofensiva olhando linha a linha. A seção 5 mostra que não é.

In [4]:
posicao = np.arange(len(df))
pacote = df["Packet_num"].to_numpy()
print("correlação Packet_num x posição da linha:", round(np.corrcoef(pacote, posicao)[0, 1], 4))
print("Sport: valores distintos / linhas =", round(df["Sport"].n_unique() / len(df), 3))

for nome, cols in [
    ("29 oficiais", FEATURES),
    ("29 + SrcMac", FEATURES + ["SrcMac"]),
    ("29 + Packet_num", FEATURES + ["Packet_num"]),
]:
    X = np.column_stack([como_numero(c) for c in cols])
    print(f"{nome:18s} AUROC = {auroc_aleatorio(X):.4f}")

correlação Packet_num x posição da linha: 1.0
Sport: valores distintos / linhas = 1.0


29 oficiais        AUROC = 0.9463


29 + SrcMac        AUROC = 1.0000


29 + Packet_num    AUROC = 0.9777


Com as 29 oficiais o modelo chega a 0,946. Somar `SrcMac` leva a 1,0 e somar `Packet_num` leva a 0,978, sem nenhuma informação nova sobre o tráfego.

**Conclusão da seção.** `SrcMac` é o rótulo. `Packet_num` e `Sport` são identificadores de posição ou de fluxo. As três saem de qualquer modelo. Daqui em diante usamos só as 29 features oficiais úteis, e tratamos `Flgs` à parte.

## 4. Split ordenado

Sinais vitais variam devagar e os ataques vêm em trechos. Num split aleatório, linhas vizinhas quase idênticas caem em treino e teste, e a métrica fica inflada.

Por isso, para a parte com janelas, cortamos o arquivo na ordem: 70% treino, 15% validação, 15% teste, com um intervalo de 50 linhas descartadas antes de cada corte. Gradient boosting não precisa de normalização.

In [5]:
n = len(df)
GAP = 50
a, b = int(0.70 * n), int(0.85 * n)

parte = np.full(n, "gap", dtype=object)
parte[: a - GAP] = "treino"
parte[a : b - GAP] = "val"
parte[b:] = "teste"

cat = pl.col("Attack Category")
df = df.with_columns(
    pl.Series("parte", parte),
    pl.col("Flgs").str.strip_chars().alias("flag"),
    (cat != cat.shift(1)).cum_sum().fill_null(0).alias("episodio"),
)

resumo = (
    df.group_by("parte")
    .agg(
        pl.len().alias("linhas"),
        (cat == "Spoofing").sum().alias("spoofing"),
        (cat == "Data Alteration").sum().alias("data_alteration"),
        pl.col("episodio").filter(cat == "Spoofing").n_unique().alias("ep_spoofing"),
        pl.col("episodio").filter(cat == "Data Alteration").n_unique().alias("ep_data_alt"),
    )
    .sort("parte")
)
print(resumo)

shape: (4, 6)
┌────────┬────────┬──────────┬─────────────────┬─────────────┬─────────────┐
│ parte  ┆ linhas ┆ spoofing ┆ data_alteration ┆ ep_spoofing ┆ ep_data_alt │
│ ---    ┆ ---    ┆ ---      ┆ ---             ┆ ---         ┆ ---         │
│ str    ┆ u32    ┆ u32      ┆ u32             ┆ u32         ┆ u32         │
╞════════╪════════╪══════════╪═════════════════╪═════════════╪═════════════╡
│ gap    ┆ 100    ┆ 0        ┆ 0               ┆ 0           ┆ 0           │
│ teste  ┆ 2448   ┆ 128      ┆ 175             ┆ 5           ┆ 7           │
│ treino ┆ 11372  ┆ 758      ┆ 698             ┆ 31          ┆ 29          │
│ val    ┆ 2398   ┆ 238      ┆ 49              ┆ 10          ┆ 2           │
└────────┴────────┴──────────┴─────────────────┴─────────────┴─────────────┘


## 5. O atalho da `Flgs`

`Flgs` vem do Argus, a ferramenta de captura usada no dataset. Cruzamos o valor (sem espaços) com o tipo de ataque.

In [6]:
cruzada = df.group_by("Attack Category", "flag").len().sort("Attack Category", "len", descending=[False, True])
print(cruzada)

print("linhas normais com R na flag:", df.filter((cat == "normal") & pl.col("flag").str.contains("R")).height)

shape: (11, 3)
┌─────────────────┬──────┬───────┐
│ Attack Category ┆ flag ┆ len   │
│ ---             ┆ ---  ┆ ---   │
│ str             ┆ str  ┆ u32   │
╞═════════════════╪══════╪═══════╡
│ Data Alteration ┆ M    ┆ 921   │
│ Data Alteration ┆ M d  ┆ 1     │
│ Spoofing        ┆ e    ┆ 984   │
│ Spoofing        ┆ eR   ┆ 138   │
│ Spoofing        ┆ M    ┆ 1     │
│ Spoofing        ┆ MR   ┆ 1     │
│ normal          ┆ e    ┆ 14253 │
│ normal          ┆ e s  ┆ 8     │
│ normal          ┆ M *  ┆ 7     │
│ normal          ┆ M    ┆ 2     │
│ normal          ┆ M d  ┆ 2     │
└─────────────────┴──────┴───────┘
linhas normais com R na flag: 0


Data Alteration é quase todo `M`. Spoofing é quase todo `e`, mas tem `eR`. O normal nunca tem `R`.

Agora olhamos onde o `R` cai dentro de cada episódio de Spoofing. Um episódio é uma sequência contígua de linhas com a mesma categoria.

In [7]:
spoof = df.filter(cat == "Spoofing").with_columns(
    (pl.col("episodio") != pl.col("episodio").shift(1)).fill_null(True).alias("primeira")
)
print("episódios de Spoofing:", spoof.filter("primeira").height)
print("flag da 1ª linha de cada episódio:")
print(spoof.filter("primeira")["flag"].value_counts().sort("count", descending=True))
print("linhas eR fora da 1ª posição do episódio:", spoof.filter(~pl.col("primeira") & (pl.col("flag") == "eR")).height)

episódios de Spoofing: 46
flag da 1ª linha de cada episódio:
shape: (2, 2)
┌──────┬───────┐
│ flag ┆ count │
│ ---  ┆ ---   │
│ str  ┆ u32   │
╞══════╪═══════╡
│ eR   ┆ 45    │
│ MR   ┆ 1     │
└──────┴───────┘
linhas eR fora da 1ª posição do episódio: 93


Pela man page do `ra` do Argus (https://www.mankier.com/1/ra):

- 1ª posição `M`: "Multiple mac addresses seen". Num man-in-the-middle a placa do atacante entra no caminho, então o fluxo passa a ter mais de um MAC.
- 2ª posição `R`: "ICMP Redirect event mapped to this flow". Quando o kernel do atacante reencaminha pacotes, ele pode mandar ICMP Redirect avisando que existe rota melhor.

O artigo do dataset diz que a captura foi feita com Argus e que o ataque usou um script Python/Scapy em Kali, com o atacante se passando pelo roteador. Ele não diz se o reencaminhamento passava pelo kernel. A explicação do `R` é, portanto, uma hipótese compatível com os dados.

O ponto prático: o `R` marca o começo do episódio. Um modelo por linha só pega essa linha. Um modelo com janela pode "lembrar" dela por várias linhas seguintes.

## 6. Ablação por tipo de ataque, com janela

Montamos janelas causais de T linhas dentro de cada parte do split ordenado. Antes do início da parte, a janela repete a primeira linha. A janela é achatada em T x features colunas.

Comparamos quatro entradas: linha atual com e sem `Flgs`, janela de 24 com e sem `Flgs`. `Flgs` entra como one-hot. Treinamos no treino e medimos no teste. A validação não é usada porque não há ajuste de hiperparâmetros. A AUROC por tipo usa só os normais do teste mais as linhas daquele tipo.

In [8]:
X_sem = df.select(FEATURES).to_numpy().astype(float)
X_com = np.hstack([X_sem, df.select("flag").to_dummies().to_numpy().astype(float)])
categoria = df["Attack Category"].to_numpy()


def janelas(X, T):
    linhas = np.arange(len(X))[:, None] - T + 1 + np.arange(T)
    return X[np.maximum(linhas, 0)].reshape(len(X), -1)


def avaliar(X, T, embaralhar=False):
    rng = np.random.default_rng(0)
    partes = {}
    for p in ["treino", "teste"]:
        idx = np.flatnonzero(parte == p)
        if embaralhar:
            idx = rng.permutation(idx)
        partes[p] = (janelas(X[idx], T), y[idx], categoria[idx])
    Xtr, ytr, _ = partes["treino"]
    Xte, yte, cte = partes["teste"]
    prob = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
    saida = {"auroc": roc_auc_score(yte, prob), "auprc": average_precision_score(yte, prob)}
    for tipo in ["Spoofing", "Data Alteration"]:
        m = (cte == "normal") | (cte == tipo)
        saida[f"auroc_{tipo}"] = roc_auc_score(yte[m], prob[m])
    return saida


ablacao = pl.DataFrame([
    {"entrada": nome, **avaliar(X, T)}
    for nome, X, T in [
        ("linha com Flgs", X_com, 1),
        ("linha sem Flgs", X_sem, 1),
        ("janela 24 com Flgs", X_com, 24),
        ("janela 24 sem Flgs", X_sem, 24),
    ]
])
print(ablacao.with_columns(pl.selectors.numeric().round(3)))

shape: (4, 5)
┌────────────────────┬───────┬───────┬────────────────┬───────────────────────┐
│ entrada            ┆ auroc ┆ auprc ┆ auroc_Spoofing ┆ auroc_Data Alteration │
│ ---                ┆ ---   ┆ ---   ┆ ---            ┆ ---                   │
│ str                ┆ f64   ┆ f64   ┆ f64            ┆ f64                   │
╞════════════════════╪═══════╪═══════╪════════════════╪═══════════════════════╡
│ linha com Flgs     ┆ 0.869 ┆ 0.749 ┆ 0.689          ┆ 1.0                   │
│ linha sem Flgs     ┆ 0.854 ┆ 0.71  ┆ 0.655          ┆ 1.0                   │
│ janela 24 com Flgs ┆ 0.995 ┆ 0.987 ┆ 0.987          ┆ 1.0                   │
│ janela 24 sem Flgs ┆ 0.94  ┆ 0.812 ┆ 0.859          ┆ 1.0                   │
└────────────────────┴───────┴───────┴────────────────┴───────────────────────┘


Data Alteration fica em 1,0 em todos os cenários. Abaixo, o motivo: a taxa de pacotes despenca durante o ataque.

In [9]:
treino_normal = df.filter((pl.col("parte") == "treino") & (cat == "normal"))
for col in ["Rate", "Load"]:
    media, desvio = treino_normal[col].mean(), treino_normal[col].std()
    media_da = df.filter(cat == "Data Alteration")[col].mean()
    print(f"{col}: normal {media:,.1f} +- {desvio:,.1f} | Data Alteration {media_da:,.1f} | diferença = {(media_da - media) / desvio:+.2f} desvios")

Rate: normal 452.9 +- 129.8 | Data Alteration 65.8 | diferença = -2.98 desvios
Load: normal 299,517.0 +- 85,814.5 | Data Alteration 31,748.8 | diferença = -3.12 desvios


Varremos o tamanho da janela com `Flgs`. No controle, embaralhamos as linhas dentro de cada parte antes de montar a janela. Assim a janela continua com 24 linhas, mas perde a ordem real.

In [10]:
varredura = pl.DataFrame([{"T": T, **avaliar(X_com, T)} for T in [1, 2, 4, 8, 24]])
print(varredura.with_columns(pl.selectors.float().round(3)))

controle = avaliar(X_com, 24, embaralhar=True)
print("controle embaralhado, T=24 com Flgs:", {k: round(v, 3) for k, v in controle.items()})

shape: (5, 5)
┌─────┬───────┬───────┬────────────────┬───────────────────────┐
│ T   ┆ auroc ┆ auprc ┆ auroc_Spoofing ┆ auroc_Data Alteration │
│ --- ┆ ---   ┆ ---   ┆ ---            ┆ ---                   │
│ i64 ┆ f64   ┆ f64   ┆ f64            ┆ f64                   │
╞═════╪═══════╪═══════╪════════════════╪═══════════════════════╡
│ 1   ┆ 0.869 ┆ 0.749 ┆ 0.689          ┆ 1.0                   │
│ 2   ┆ 0.921 ┆ 0.809 ┆ 0.814          ┆ 1.0                   │
│ 4   ┆ 0.962 ┆ 0.881 ┆ 0.909          ┆ 1.0                   │
│ 8   ┆ 0.981 ┆ 0.931 ┆ 0.956          ┆ 1.0                   │
│ 24  ┆ 0.995 ┆ 0.987 ┆ 0.987          ┆ 1.0                   │
└─────┴───────┴───────┴────────────────┴───────────────────────┘


controle embaralhado, T=24 com Flgs: {'auroc': 0.893, 'auprc': 0.773, 'auroc_Spoofing': 0.747, 'auroc_Data Alteration': 1.0}


## 7. Conclusão

- **Vazamento.** `SrcMac` é o rótulo. `Packet_num` e `Sport` são identificadores. Nenhuma das três pode entrar no modelo.
- **Data Alteration é trivial.** `Rate` e `Load` caem cerca de 3 desvios durante o ataque, e as quatro entradas chegam a AUROC 1,0.
- **O ganho da janela no Spoofing é em boa parte atalho.** O Spoofing quase sempre começa com `eR` (ICMP Redirect). Na janela de 24, a AUROC de Spoofing é 0,987 com `Flgs` e 0,859 sem. Por linha a diferença é pequena (0,689 contra 0,655), porque só a 1ª linha do episódio tem o `R`. O modelo com janela aprende a lembrar essa linha.
- **O ganho depende da ordem real.** Com as linhas embaralhadas dentro de cada parte, a janela de 24 com `Flgs` cai de 0,995 para 0,893 de AUROC geral.
- **Cenário comparável à literatura.** `Flgs` está fora da Tabela 1 do dataset. O resultado comparável é o das 29 features oficiais úteis: 0,854 por linha e 0,940 com janela de 24, no split ordenado.
- **O atalho é frágil.** Um atacante evita o redirect com `net.ipv4.conf.all.send_redirects=0` ou encaminhando os pacotes em espaço de usuário.
- **Incerteza.** O teste ordenado tem só 12 episódios de ataque (5 de Spoofing e 7 de Data Alteration, tabela da seção 4). A AUROC é calculada por linha, mas as linhas de um episódio não são independentes, então a incerteza real é bem maior do que parece.
- **Um só atacante.** O dataset tem um único atacante e uma única ferramenta. Nada aqui diz como o modelo se sairia contra outro.